In [126]:
import wandb
from bibliotecas_externas.seq2seqvc.seq2seq_vc.losses import Seq2SeqLoss, DurationPredictorLoss, \
    StochasticDurationPredictorLoss

from main.arquitetura.Seq2SeqVC.models.model import seq2seq_AASVC
from main.dataloader.CVMPT.CVMPT_offline import CVMPT_offline
import torch



In [127]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [128]:
yaml_model = r"C:\Users\USER\Documents\Mestrado\codigo\Mestrado_VC\main\arquitetura\Seq2SeqVC\configs\AASVC_ENG\aas_vc.melmelmel.v1.yaml"
checkpoint_model_path = r"C:\Users\USER\Documents\Mestrado\codigo\Mestrado_VC\main\arquitetura\Seq2SeqVC\configs\AASVC_JP\checkpoint-50000steps.pkl"

dataset_train_path = r"C:\Users\USER\Documents\Mestrado\codigo\Mestrado_VC\dataset\cv-corpus-mozilla-pt\data\Treinamento_gp"
dataset_val_path = r"C:\Users\USER\Documents\Mestrado\codigo\Mestrado_VC\dataset\cv-corpus-mozilla-pt\data\Teste_gp"

In [129]:
from main.arquitetura.Seq2SeqVC.datasets.collaters.nar_vc import NARVCCollater

device = "cuda" if torch.cuda.is_available() else "cpu"
model_seq2seq = seq2seq_AASVC(yaml_model,device)

optimizer = torch.optim.Adam(
    model_seq2seq.parameters(),
    lr=1e-4,
    betas=(0.9, 0.98),
    eps=1e-9
)

l1_loss = torch.nn.L1Loss()
bce_loss = torch.nn.BCEWithLogitsLoss()


train_set = CVMPT_offline(path=dataset_train_path)
valid_set = CVMPT_offline(path=dataset_val_path)

train_loader = torch.utils.data.DataLoader(
    train_set,
    batch_size=2,
    shuffle=True,
    collate_fn=NARVCCollater()
)

valid_loader = torch.utils.data.DataLoader(
    valid_set,
    batch_size=1,
    shuffle=False,
    collate_fn=NARVCCollater()
)


In [130]:
from bibliotecas_externas.seq2seqvc.seq2seq_vc.vocoder.griffin_lim import Spectrogram2Waveform

vocoder = Spectrogram2Waveform(
    n_fft=1024,
    n_shift=256,
    fs=22050,
    n_mels=80,
    griffin_lim_iters=32,
    take_norm_feat=False,
    #stats=trg_stats,  # stats do target
)


In [131]:
from main.arquitetura.Seq2SeqVC.trainers.TrainerAASVCMod import Trainer
from bibliotecas_externas.seq2seqvc.seq2seq_vc.losses import L1Loss, ForwardSumLoss
from bibliotecas_externas.seq2seqvc.seq2seq_vc.trainers import ARVCTrainer, AASVCTrainer

# contadores
steps = 0
epochs = 0

# dataloaders
data_loader = {
    "train": train_loader,
    "dev": valid_loader,
}

# sampler (sem DDP)
sampler = {}

# modelo
model = model_seq2seq.to(device)

# loss
criterion = {
    "L1Loss": L1Loss(),
    "ForwardSumLoss": ForwardSumLoss(),
    "StochasticDurationPredictorLoss": StochasticDurationPredictorLoss(),
}


# scheduler (opcional)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=4000,
    gamma=1.0
)

inference_args = {
    "threshold": 0.5,
    "minlenratio": 6.0,
    "maxlenratio": 0.0,
}

# config
config = {
    "outdir": "./experiments/aas_vc_test_inference",
    "train_max_steps": 60000,
    "log_interval_steps": 10,
    "eval_interval_steps": 100,
    "save_interval_steps": 5000,
    "distributed": False,
    "rank": 0,
    "gradient_accumulate_steps": 1,
    "grad_norm": 0,
    "num_save_intermediate_results": 3,
    "inference": inference_args,
    "criterions": ["L1Loss", "ForwardSumLoss", "StochasticDurationPredictorLoss"],
    "lambda_align": 2.0,
    "dp_train_start_steps":0
    
}

# trainer
trainer = Trainer(
    steps=steps,
    epochs=epochs,
    data_loader=data_loader,
    sampler=sampler,
    model=model,
    vocoder=vocoder,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    config=config,
    is_test=True,
    device=device,
)

trainer.load_checkpoint(checkpoint_model_path)

In [132]:
project="Laringe Eletronica Seq2Seq - Mestrado VC"

config = {
    "epocas": 100
}
trainer.run_wandb(project,config)

[train]:  83%|########3 | 50000/60000 [00:00<?, ?it/s]

Validacao:   0%|          | 0/9579 [00:00<?, ?it/s]

HiFi-GAN: Removing weight norm.


Validacao:   0%|          | 0/9579 [00:00<?, ?it/s]

HiFi-GAN: Removing weight norm.


...

HiFi-GAN: Removing weight norm.


Validacao:   0%|          | 0/9579 [00:00<?, ?it/s]

HiFi-GAN: Removing weight norm.


Traceback (most recent call last):
  File "C:\Users\USER\Documents\Mestrado\codigo\Mestrado_VC\main\arquitetura\Seq2SeqVC\trainers\Seq2SeqTrainerMod.py", line 47, in run_wandb
  File "C:\Users\USER\Documents\Mestrado\codigo\Mestrado_VC\.venv\Lib\site-packages\wandb\sdk\wandb_run.py", line 400, in wrapper
    return func(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\USER\Documents\Mestrado\codigo\Mestrado_VC\.venv\Lib\site-packages\wandb\sdk\wandb_run.py", line 458, in wrapper_fn
    return func(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\USER\Documents\Mestrado\codigo\Mestrado_VC\.venv\Lib\site-packages\wandb\sdk\wandb_run.py", line 445, in wrapper
    return func(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\USER\Documents\Mestrado\codigo\Mestrado_VC\.venv\Lib\site-packages\wandb\sdk\wandb_run.py", line 2034, in log
    self._log(data=data, step=step, commit=commit)
  File "C:\Users\US

TypeError: wandb.log must be passed a dictionary

In [ ]:
# from main.arquitetura.Seq2SeqVC.deprected.train import one_epoch
# 
# criterion = L1Loss()
# 
# project="Laringe Eletronica Seq2Seq - Mestrado VC"
# config = {
#     "epocas": 5
# }
# 
# print("-------------------------------------------")
# print(f"Trenamento Modelo: {project}")
# with wandb.init(project=project,config=config) as run:
# 
#     for ep in range(config["epocas"]):
#         print(f"Epoca: {ep+1}")
#         r = one_epoch(model_seq2seq, device, train_loader,valid_loader, optimizer, criterion, ep, is_test=False)
#         run.log(r)
#         print("-------------------------------------------")


In [2]:
import os
import imageio
import re
pasta_raiz = r"C:\Users\USER\Documents\Mestrado\codigo\Mestrado_VC\main\arquitetura\Seq2SeqVC\experiments\aas_vc_test_inference\predictions"



# ordenar as pastas (epocas)
epocas = sorted([
    p for p in os.listdir(pasta_raiz)
    if os.path.isdir(os.path.join(pasta_raiz, p))
])

# pegar nomes das imagens da primeira epoca
primeira_epoca = os.path.join(pasta_raiz, epocas[0])
imagens = [f for f in os.listdir(primeira_epoca) if f.endswith(".png")]

for img_nome in imagens:

    frames = []

    for epoca in epocas:

        caminho = os.path.join(pasta_raiz, epoca, img_nome)

        if os.path.exists(caminho):
            frames.append(imageio.imread(caminho))

    if frames:
        imageio.mimsave(f"{img_nome.replace('.png','')}.gif", frames, duration=0.5)

        print(f"GIF criado: {img_nome}")

C:\Users\USER\AppData\Local\Temp\ipykernel_9524\3315135417.py:27: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  frames.append(imageio.imread(caminho))


GIF criado: 0_out.png
